# 02B — Frozen-backbone feature extraction (PyTorch, GPU)

Extracts frozen features for the nine PyTorch configurations of the paper (Table 1). ResNet50 is
handled separately in notebook 02.

| `MODEL_NAME` | Weights | Input | Feature |
|---|---|---|---|
| `clip_vitl14` | open_clip `ViT-L-14`, `openai` | 224 | 768 |
| `clip_vith14` | open_clip `ViT-H-14`, `laion2b_s32b_b79k` | 224 | 1024 |
| `siglip2_so400m` (main model) | timm `vit_so400m_patch16_siglip_384.v2_webli` | 384 | 1152 |
| `siglip2_so400m_224` | timm `vit_so400m_patch14_siglip_224.v2_webli` | 224 | 1152 |
| `dinov2_vitl14` | timm `vit_large_patch14_dinov2.lvd142m` | 224 | 1024 |
| `dinov3_vitl16` | timm `vit_large_patch16_dinov3.lvd1689m` (bfloat16) | 256 | 1024 |
| `convnext_l_in22k` | timm `convnext_large.fb_in22k` | 224 | 1536 |
| `convnext_l_clip` | timm `convnext_large_mlp.clip_laion2b_augreg` | 256 | 1536 |
| `convnext_l_clip_320` | timm `convnext_large_mlp.clip_laion2b_ft_320` | 320 | 1536 |

**Preprocessing** (Section IV-A): images whose long side exceeds 2048 px are first downscaled to this size
(bicubic); each view is then resized to the model's square input **without preserving the aspect ratio**
(bicubic) and normalised with the model's own mean and standard deviation.

**Views** (`VIEWS`): `full` is the whole image. `full+crops` adds deterministic crops for the multi-view
experiments (Section IV-E, Fig. 2):
- `CROP_MODE="grid_square"`, `N_CROPS=6`, `CROP_FRAC=0.5`: six square windows whose side is half the short
  side, on a regular 2 × 3 grid (3 × 2 for portrait images);
- `CROP_MODE="quadrants"`, `N_CROPS=4`: the four quadrants of a 2 × 2 grid.

Features are L2-normalised and stored as float16. Computation is in fp16 (bfloat16 for DINOv3).

**Environment:** PyTorch + CUDA (see `requirements-gpu.txt`).

**Output** (`cache/embeddings/gpu/`): `<TAG>.npy` with shape (N, V, D), `<TAG>_index.parquet`,
`<TAG>_info.json`, e.g. `TAG = wikiart_siglip2_so400m_full`.

The extraction is written in chunks; if it is interrupted, re-running the extraction cell resumes.

In [ ]:
import importlib, platform
print("Python", platform.python_version())
for mod in ["torch", "timm", "open_clip", "torchvision", "numpy", "PIL"]:
    try:
        m = importlib.import_module(mod)
        print(f"{mod:12s}", getattr(m, "__version__", "?"))
    except ImportError:
        print(f"{mod:12s}", "not installed")

In [ ]:
import json, time, hashlib, warnings
from collections import deque
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
from PIL import Image, ImageFile
from tqdm.auto import tqdm
import torch

warnings.filterwarnings("ignore")
ImageFile.LOAD_TRUNCATED_IMAGES = True
Image.MAX_IMAGE_PIXELS = None

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Device :", torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU (slow; check the GPU environment)")

## Configuration

In [ ]:
import os
from pathlib import Path

# Project root: the repository folder (notebooks/ is one level below it).
# Override with the environment variable ARTEMIS_ROOT if the data live elsewhere.
PROJECT_ROOT = Path(os.environ.get("ARTEMIS_ROOT", Path.cwd().parent)).resolve()

DATA_DIR      = PROJECT_ROOT / "data"
WIKIART_DIR   = DATA_DIR / "wikiart"
METADATA_DIR  = DATA_DIR / "metadata"
GPU_DIR       = PROJECT_ROOT / "cache" / "embeddings" / "gpu"
CHUNK_DIR     = GPU_DIR / "_chunks"
for d in (GPU_DIR, CHUNK_DIR):
    d.mkdir(parents=True, exist_ok=True)

METADATA_FILE           = METADATA_DIR / "artemis_metadata.parquet"
SELECTED_MOVEMENTS_FILE = METADATA_DIR / "selected_movements.json"

# ---------------- What to extract ----------------
DATASET    = "wikiart"         # used in the file tag only
MODEL_NAME = "siglip2_so400m"  # one of the keys of MODELS below
VIEWS      = "full"            # "full" | "full+crops" (multi-view experiments, CLIP ViT-L/14 only in the paper)
CROP_MODE  = "grid_square"     # "grid_square" | "quadrants"
N_CROPS    = 6                 # grid_square: 6 in the paper; quadrants: 4
CROP_FRAC  = 0.5               # grid_square: window side = CROP_FRAC x short side

# ---------------- Performance ----------------
BATCH_IMAGES = 16              # images per batch (each produces V views); lower if GPU memory runs out
LOAD_THREADS = 8               # image loading / cropping threads
CHUNK_SIZE   = 2048            # images per chunk on disk
MAX_SIDE     = 2048            # larger images (long side) are first downscaled to this size
USE_FP16     = True            # half-precision inference on the GPU
PILOT_N      = None            # e.g. 200: quick test on the first 200 images (separate file tag)

CLIP_MEAN, CLIP_STD = (0.48145466, 0.4578275, 0.40821073), (0.26862954, 0.26130258, 0.27577711)
IMNET_MEAN, IMNET_STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
MODELS = {
    "clip_vitl14":   dict(kind="open_clip", arch="ViT-L-14", pretrained="openai", mean=CLIP_MEAN, std=CLIP_STD),
    "clip_vith14":   dict(kind="open_clip", arch="ViT-H-14", pretrained="laion2b_s32b_b79k", mean=CLIP_MEAN, std=CLIP_STD),
    "siglip2_so400m": dict(kind="timm", name="vit_so400m_patch16_siglip_384.v2_webli", input_size=384,
                           mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
    "siglip2_so400m_224": dict(kind="timm", name="vit_so400m_patch14_siglip_224.v2_webli", input_size=224,
                           mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
    "dinov2_vitl14": dict(kind="timm", name="vit_large_patch14_dinov2.lvd142m", mean=IMNET_MEAN, std=IMNET_STD),
    # DINOv3 overflows in fp16, so it runs in bfloat16; the CLS token is used (as for DINOv2)
    "dinov3_vitl16": dict(kind="timm", name="vit_large_patch16_dinov3.lvd1689m", input_size=256, dtype="bf16",
                          mean=IMNET_MEAN, std=IMNET_STD, kwargs=dict(global_pool="token")),
    "convnext_l_in22k": dict(kind="timm", name="convnext_large.fb_in22k", input_size=224, img_size_kw=False,
                          mean=IMNET_MEAN, std=IMNET_STD),
    "convnext_l_clip": dict(kind="timm", name="convnext_large_mlp.clip_laion2b_augreg", input_size=256,
                          img_size_kw=False, mean=CLIP_MEAN, std=CLIP_STD),
    "convnext_l_clip_320": dict(kind="timm", name="convnext_large_mlp.clip_laion2b_ft_320", input_size=320,
                          img_size_kw=False, mean=CLIP_MEAN, std=CLIP_STD),
}
SPEC = MODELS[MODEL_NAME]
INPUT_SIZE = SPEC.get("input_size", 224)
GRID = {4: (2, 2), 6: (2, 3), 9: (3, 3)}     # (rows, columns) for landscape images; swapped for portrait
assert VIEWS in ("full", "crops", "full+crops")
assert N_CROPS in GRID and CROP_MODE in ("grid_square", "quadrants")
if CROP_MODE == "quadrants":
    assert N_CROPS == 4, 'CROP_MODE="quadrants" requires N_CROPS=4'

VIEW_NAMES = (["full"] if VIEWS in ("full", "full+crops") else []) + \
             ([f"crop{i}" for i in range(N_CROPS)] if VIEWS in ("crops", "full+crops") else [])
N_VIEWS = len(VIEW_NAMES)

_vtag = {"full": "full", "crops": f"crops{N_CROPS}", "full+crops": f"fullcrops{N_CROPS}"}[VIEWS]
_ctag = f"_f{CROP_FRAC:g}" if CROP_MODE == "grid_square" else "_quad"
TAG = f"{DATASET}_{MODEL_NAME}_{_vtag}" + (_ctag if VIEWS != "full" else "") + (f"_pilot{PILOT_N}" if PILOT_N else "")
FEATURE_FILE = GPU_DIR / f"{TAG}.npy"
INDEX_FILE   = GPU_DIR / f"{TAG}_index.parquet"
INFO_FILE    = GPU_DIR / f"{TAG}_info.json"
print(f"MODEL={MODEL_NAME}  VIEWS={VIEW_NAMES}  CROP_MODE={CROP_MODE}  TAG={TAG}")

## Image list (the 20 selected movements)

In [ ]:
meta = pd.read_parquet(METADATA_FILE)
with open(SELECTED_MOVEMENTS_FILE, encoding="utf-8") as f:
    SELECTED = json.load(f)["movements"]
meta = meta[meta["movement"].isin(SELECTED)]
names = meta["filename"].astype(str).tolist()

paths = [str(WIKIART_DIR / n) for n in names]
exists = np.array([Path(p).is_file() for p in paths])
if not exists.all():
    print(f"⚠ {int((~exists).sum()):,} files not found on disk; skipped.")
names = [n for n, e in zip(names, exists) if e]
paths = [p for p, e in zip(paths, exists) if e]
if PILOT_N:
    names, paths = names[:PILOT_N], paths[:PILOT_N]
N = len(paths)
print(f"Images to process: {N:,}")   # 65,998
assert N > 0

## Model, views and encoder

In [ ]:
def load_model():
    if SPEC["kind"] == "open_clip":
        import open_clip
        m, _, _ = open_clip.create_model_and_transforms(SPEC["arch"], pretrained=SPEC["pretrained"])
        enc = m.encode_image
    else:
        import timm
        kw = dict(SPEC.get("kwargs", {}))
        if SPEC.get("img_size_kw", True):          # ConvNeXt does not take img_size
            kw["img_size"] = INPUT_SIZE
        m = timm.create_model(SPEC["name"], pretrained=True, num_classes=0, **kw)
        enc = m
    m = m.to(device).eval()
    want = SPEC.get("dtype", "fp16") if (USE_FP16 and device.type == "cuda") else "fp32"
    if want == "bf16" and not torch.cuda.is_bf16_supported():
        want = "fp32"
    cdt = {"fp16": torch.float16, "bf16": torch.bfloat16, "fp32": torch.float32}[want]
    if cdt != torch.float32:
        m = m.to(cdt)
    return m, enc, cdt, want

model, encode, CDTYPE, CDTYPE_NAME = load_model()
HALF = CDTYPE != torch.float32
MEAN = torch.tensor(SPEC["mean"], device=device).view(1, 3, 1, 1)
STD  = torch.tensor(SPEC["std"],  device=device).view(1, 3, 1, 1)


def crop_boxes(w, h):
    """Deterministic crop windows. grid_square: square windows on a regular grid;
    quadrants: the four equal quadrants of the image."""
    if CROP_MODE == "quadrants":
        xm, ym = w // 2, h // 2
        return [(0, 0, xm, ym), (xm, 0, w, ym), (0, ym, xm, h), (xm, ym, w, h)]
    s = max(8, int(round(CROP_FRAC * min(w, h))))
    rows, cols = GRID[N_CROPS]
    if w < h:                       # portrait image: more rows
        rows, cols = cols, rows
    xs = np.linspace(0, w - s, cols).round().astype(int)
    ys = np.linspace(0, h - s, rows).round().astype(int)
    return [(int(x), int(y), int(x) + s, int(y) + s) for y in ys for x in xs]


def load_views(path):
    """Returns a (V, S, S, 3) uint8 array, or None if the image cannot be read."""
    try:
        img = Image.open(path).convert("RGB")
        if max(img.size) > MAX_SIDE:
            img.thumbnail((MAX_SIDE, MAX_SIDE), Image.BICUBIC)
        w, h = img.size
        out = []
        if VIEWS in ("full", "full+crops"):
            out.append(img.resize((INPUT_SIZE, INPUT_SIZE), Image.BICUBIC))
        if VIEWS in ("crops", "full+crops"):
            for box in crop_boxes(w, h):
                out.append(img.crop(box).resize((INPUT_SIZE, INPUT_SIZE), Image.BICUBIC))
        return np.stack([np.asarray(v, dtype=np.uint8) for v in out])
    except Exception:
        return None


def iter_views(path_list):
    """Generator that preloads images with a thread pool and bounded memory."""
    ex = ThreadPoolExecutor(LOAD_THREADS)
    dq, nxt, ahead = deque(), 0, LOAD_THREADS * 4
    def fill():
        nonlocal nxt
        while nxt < len(path_list) and len(dq) < ahead:
            dq.append(ex.submit(load_views, path_list[nxt])); nxt += 1
    fill()
    try:
        while dq:
            f = dq.popleft(); fill()
            yield f.result()
    finally:
        ex.shutdown(wait=False)


@torch.inference_mode()
def encode_batch(arrs):
    """arrs: list of (V,S,S,3) uint8 -> (B, V, D) float32, L2-normalised."""
    x = torch.from_numpy(np.concatenate(arrs, axis=0)).to(device)
    x = x.permute(0, 3, 1, 2).float().div_(255.0)
    x = (x - MEAN) / STD
    if HALF:
        x = x.to(CDTYPE)
    f = encode(x).float()
    if not torch.isfinite(f).all():
        raise FloatingPointError(f"{MODEL_NAME}: NaN/Inf in {CDTYPE_NAME}; use dtype='bf16' or 'fp32' for this model "
                                 "and delete its chunk files.")
    f = f / f.norm(dim=-1, keepdim=True).clamp_min(1e-8)
    return f.view(len(arrs), N_VIEWS, -1).cpu().numpy()


_probe = next(v for v in (load_views(p) for p in paths) if v is not None)
D = encode_batch([_probe]).shape[-1]
print(f"Feature dimension D = {D}   views V = {N_VIEWS}   ({CDTYPE_NAME})")

## Extraction (in chunks; resumes after interruption)

In [ ]:
n_chunks = (N + CHUNK_SIZE - 1) // CHUNK_SIZE
t_start, done_imgs = time.time(), 0

for k in range(n_chunks):
    a, b = k * CHUNK_SIZE, min(N, (k + 1) * CHUNK_SIZE)
    h = hashlib.md5("\n".join(names[a:b]).encode("utf-8")).hexdigest()[:8]
    cf = CHUNK_DIR / f"{TAG}_{k:04d}_{h}.npz"
    if cf.exists():
        with np.load(cf) as z:
            good = np.isfinite(z["feats"][z["ok"]].astype(np.float32)).all()
        if good:
            done_imgs += b - a
            continue
        cf.unlink()
    feats = np.zeros((b - a, N_VIEWS, D), np.float16)
    ok = np.zeros(b - a, bool)
    buf, buf_i = [], []
    def flush():
        global buf, buf_i
        if buf:
            feats[buf_i] = encode_batch(buf).astype(np.float16)
            ok[buf_i] = True
            buf, buf_i = [], []
    for j, arr in enumerate(tqdm(iter_views(paths[a:b]), total=b - a, desc=f"chunk {k + 1}/{n_chunks}", leave=False)):
        if arr is None:
            continue
        buf.append(arr); buf_i.append(j)
        if len(buf) == BATCH_IMAGES:
            flush()
    flush()
    np.savez(cf, feats=feats, ok=ok)
    done_imgs += b - a
    print(f"✓ chunk {k + 1}/{n_chunks} | {(time.time() - t_start) / 60:.1f} min | {done_imgs:,}/{N:,} images")
print("Extraction finished.")

## Merge chunks, write index and specification

In [ ]:
mm = np.lib.format.open_memmap(FEATURE_FILE, mode="w+", dtype=np.float16, shape=(N, N_VIEWS, D))
ok_all = np.zeros(N, bool)
for k in range(n_chunks):
    a, b = k * CHUNK_SIZE, min(N, (k + 1) * CHUNK_SIZE)
    h = hashlib.md5("\n".join(names[a:b]).encode("utf-8")).hexdigest()[:8]
    z = np.load(CHUNK_DIR / f"{TAG}_{k:04d}_{h}.npz")
    mm[a:b] = z["feats"]
    ok_all[a:b] = z["ok"]
mm.flush(); del mm

print(f"Unreadable images: {int((~ok_all).sum())}")
pd.DataFrame(dict(filename=names, embedding_index=np.arange(N), load_ok=ok_all)).to_parquet(INDEX_FILE, index=False)

info = dict(tag=TAG, dataset=DATASET, model_name=MODEL_NAME, library=SPEC["kind"],
            arch=SPEC.get("arch") or SPEC.get("name"),
            pretrained=SPEC.get("pretrained") or SPEC["name"].split(".")[-1],
            input_size=INPUT_SIZE, mean=list(SPEC["mean"]), std=list(SPEC["std"]),
            full_resize=f"squash to {INPUT_SIZE}x{INPUT_SIZE} without preserving the aspect ratio (bicubic)",
            crop_resize=(f"square window -> {INPUT_SIZE}x{INPUT_SIZE} (bicubic)" if CROP_MODE == "grid_square"
                         else f"rectangular quadrant -> {INPUT_SIZE}x{INPUT_SIZE} (bicubic, aspect ratio not preserved)"),
            views=VIEWS, view_names=VIEW_NAMES, n_crops=N_CROPS if VIEWS != "full" else 0,
            crop_mode=CROP_MODE, crop_frac=CROP_FRAC if CROP_MODE == "grid_square" else None,
            crop_grid=GRID[N_CROPS], max_side=MAX_SIDE,
            l2_normalized=True, stored_dtype="float16",
            compute_dtype={"fp16": "float16", "bf16": "bfloat16", "fp32": "float32"}[CDTYPE_NAME],
            shape=[N, N_VIEWS, D], n_images=N, n_failed=int((~ok_all).sum()),
            torch=torch.__version__, created_at=datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
with open(INFO_FILE, "w", encoding="utf-8") as f:
    json.dump(info, f, ensure_ascii=False, indent=2)
print("✓", FEATURE_FILE.name, (N, N_VIEWS, D))

## Verification (numerical health and re-extraction consistency)

In [ ]:
F = np.load(FEATURE_FILE, mmap_mode="r")
good = np.where(pd.read_parquet(INDEX_FILE)["load_ok"].to_numpy())[0]

sample = np.sort(np.random.default_rng(0).choice(good, size=min(500, len(good)), replace=False))
S = np.asarray(F[sample]).astype(np.float32)
norms = np.linalg.norm(S, axis=-1)
print(f"NaN/Inf present : {not np.isfinite(S).all()}")
print(f"Norm range      : {norms.min():.4f} – {norms.max():.4f}  (≈1 expected)")

vs = np.random.default_rng(1).choice(good, size=min(50, len(good)), replace=False)
fresh = np.stack([encode_batch([load_views(paths[i])])[0] for i in vs])[np.argsort(vs)]
stored = np.asarray(F[np.sort(vs)]).astype(np.float32)
cos = (fresh * stored).sum(-1) / (np.linalg.norm(fresh, axis=-1) * np.linalg.norm(stored, axis=-1))
print(f"Re-extraction cosine: min {cos.min():.5f}  mean {cos.mean():.5f}  (>0.99 expected with fp16 storage)")
if N_VIEWS > 1:
    print(f"View 0 vs 1 mean cosine: {(S[:, 0] * S[:, 1]).sum(-1).mean():.3f}  (1.0 would mean the crops are not working)")
assert np.isfinite(S).all() and (norms > 1e-3).all() and cos.min() > 0.99
print("✓ Verification passed")